In [ ]:
%pip install websockets



[notice] A new release of pip is available: 24.0 -> 24.2
[notice] To update, run: pip install --upgrade pip


In [ ]:
import json
import websockets

# Move codes, the rover's address, the no-echo distance and the telemetry key
# for each bearing come from drive.py, beside this notebook in client/, where
# tools/check_protocol.py keeps them in step with the firmware. A copy kept here
# would drift unseen: a hand-copied 16 was sent as "clockwise"; 16 is
# PIVOT_SIDEWAYS_BACKWARD_LEFT. Jupyter and VS Code start the kernel in the
# notebook's own folder, which is how this import finds drive.py.
from drive import BEARINGS, DEFAULT_HOST, DEFAULT_PORT, DISTANCE_FAR_CM, MOVE_FORWARD, ROTATE_CLOCKWISE

# Frames in a row with no echo at any bearing before this loop gives up.
# Telemetry comes every 500 ms, so 6 is about 3 s: roughly the three silent
# sweeps after which the rover's own exploration halts with "sensor silent".
SILENT_FRAMES_TO_GIVE_UP = 6

async def websocket_communication():
    uri = f"ws://{DEFAULT_HOST}:{DEFAULT_PORT}"  # Replace with your rover's address

    # Every command switches the rover to manual mode. To let it explore again
    # afterwards, send RESUME_AUTONOMOUS (or press t in drive.py).
    async with websockets.connect(uri) as websocket:
        silent_frames = 0
        try:
            while True:
              response = await websocket.recv()
              response = json.loads(response)
              print(f"{response}")
              # Distances appear only once the scanner has measured every
              # bearing, so the first frames after a reset carry none.
              if "distanceFront" not in response:
                continue

              # 999 (DISTANCE_FAR_CM) means no echo came back, which is not a
              # measurement of open space: a dead or unplugged sensor reads 999
              # at every bearing. So drive forward only once some bearing has
              # heard an echo; with none, turn in place to look again, as the
              # rover's own exploration does, and give up if the silence lasts.
              heard = any(response.get(key, DISTANCE_FAR_CM) < DISTANCE_FAR_CM for _, key in BEARINGS)
              silent_frames = 0 if heard else silent_frames + 1
              if silent_frames >= SILENT_FRAMES_TO_GIVE_UP:
                # The last 300 ms command ran out before this frame arrived,
                # and leaving the `async with` closes the link, which stops a
                # rover this loop was driving anyway.
                print("No echo at any bearing for about 3 s: check the sonar. Giving up.")
                break

              request = {
                  "speed": 75,
                  "duration": 300
              }
              if not heard or response["distanceFront"] < 20:
                request["move"] = ROTATE_CLOCKWISE
                request["speed"] = 128
              else:
                request["move"] = MOVE_FORWARD

              request = json.dumps(request)
              print(f"{request}")
              await websocket.send(request)
              # No sleep: telemetry arrives every 500 ms and paces this loop.
              # Sleeping longer lets frames queue faster than they are read, so
              # every decision is made on older data, and once the queue fills
              # the library stops reading -- the rover's heartbeat pings go
              # unanswered and it drops the connection.
        except websockets.ConnectionClosed as e:
            print(f"Connection closed: {e}")

# Running the WebSocket client in a loop in a Jupyter notebook
await websocket_communication()
